# Iterative + Reflection RAG Pipeline (Deep Explanation)

---

## Concept Name
**Iterative Retrieval with Self‑Reflection RAG**

This pipeline is a specialized form of **Retrieval‑Augmented Generation (RAG)**.  
It combines two advanced strategies:
- **Iterative Retrieval**: The agent does not settle for the first batch of retrieved documents. If the answer is incomplete, it refines the query and retrieves again.  
- **Self‑Reflection**: The agent evaluates its own answer to decide whether it is factually sufficient and complete.  

Together, these strategies create a **self‑correcting loop** that improves answer quality and transparency.

---

## Code Flow (Step by Step)

### 1. Knowledge Base Preparation
- The Xumo manual text file is loaded.  
- Text is split into manageable chunks.  
- Each chunk is embedded using Hugging Face embeddings.  
- Stored in a FAISS vector store for fast similarity search.  

### 2. LLM Initialization
- Groq LLM (`llama‑3.1‑8b‑instant`) is used for answering, reflecting, and refining queries.  
- This model is responsible for generating answers, evaluating them, and suggesting refined queries.

### 3. Agent State
The pipeline maintains a state object that tracks:
- The original question.  
- Any refined query (if reflection finds the answer incomplete).  
- Retrieved documents.  
- The generated answer.  
- Whether the answer was verified as sufficient.  
- The number of attempts made.  

This state ensures continuity across iterations.

---

## Graph Nodes

### a. Retrieve Node
- Uses either the refined query or the original question.  
- Retrieves relevant chunks from FAISS.  
- Updates the state with retrieved documents.  
- This ensures that each iteration has fresh context.

### b. Generate Answer Node
- Builds a prompt using the retrieved context.  
- LLM generates an answer based on this context.  
- Increments the attempt counter.  
- This step produces the candidate answer for evaluation.

### c. Reflect Node
- LLM evaluates its own answer.  
- Prompt asks: *“Is this answer factually sufficient and complete?”*  
- If the response contains “YES”, the answer is marked verified.  
- If “NO”, the pipeline knows the answer is incomplete.  
- This introduces self‑critique into the workflow.

### d. Refine Query Node
- If reflection says the answer is incomplete, the LLM suggests a better query.  
- This refined query is used in the next retrieval step.  
- Ensures that the agent can adapt its search strategy dynamically.

---

## Graph Flow

[Retrieve] → [Answer] → [Reflect]

↘

↘ if NO → [Refine] → [Retrieve]



- Entry point: Retrieve.  
- After answering, pipeline goes to Reflect.  
- If reflection = YES → End.  
- If reflection = NO → Refine query → Loop back to Retrieve.  
- Maximum of 2 attempts → End anyway.  

---

## Example Run

**User Query:**  
“Does the Xumo Stream Box support 4K and Dolby Atmos?”

**Execution:**
1. Retrieve → Finds HDMI + Dolby Vision + Dolby Atmos chunks.  
2. Answer → “Dolby Vision suggests possible 4K, Atmos supported.”  
3. Reflect → NO (incomplete).  
4. Refine → New query: “Xumo Stream Box resolution specifications.”  
5. Retrieve again → Finds resolution‑related chunks.  
6. Answer → “Manual does not explicitly confirm 4K, but Dolby Vision HDR implies 4K capability.”  
7. Reflect → NO again, but attempts = 2 → Stop.  

**Final Output:**
- Answer: “Manual does not explicitly confirm 4K. Dolby Vision HDR and Dolby Atmos are supported.”  
- Verified: False  
- Attempts: 2  

---

## Why This Matters
- Iterative Retrieval ensures the agent does not stop at the first attempt.  
- Self‑Reflection introduces a feedback loop, improving reliability.  
- The bounded loop (maximum 2 attempts) prevents infinite retries.  
- Transparency is built in: the user sees the final answer, reflection verdict, and attempt count.  

---

## Takeaway
This pipeline is called **Iterative + Reflection RAG**.  
It is a **self‑correcting retrieval pipeline** that retrieves, answers, reflects, refines, and retries until it is confident enough to finalize.  
It represents a more human‑like research process, where answers are checked, refined, and improved iteratively.

In [3]:
# ---------------------------------
# 0. Setup & Imports
# ---------------------------------
import os
from typing import List
from pydantic import BaseModel
from dotenv import load_dotenv
import gradio as gr

from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

In [4]:
# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# ---------------------------------
# 1. Load and Embed Documents
# ---------------------------------
manual = TextLoader(
    "C:/Users/admin/Desktop/New_GenAI/GenAI/LangGraph/Autonomus RAG/xumo_manual_rag.txt",
    encoding="utf-8"
).load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(manual)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

In [5]:
# ---------------------------------
# 2. Initialize Groq LLM
# ---------------------------------
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [ ]:
# ---------------------------------
# 3. Define Agent State
# ---------------------------------
class IterativeReflectRAGState(BaseModel):
    question: str
    refined_question: str = ""
    retrieved_docs: List[Document] = []
    answer: str = ""
    verified: bool = False
    attempts: int = 0
    

In [7]:
# ---------------------------------
# 4. Nodes
# ---------------------------------

# a. Retrieve
def retrieve_docs(state: IterativeReflectRAGState) -> IterativeReflectRAGState:
    query = state.refined_question or state.question
    docs = retriever.invoke(query)
    return state.model_copy(update={"retrieved_docs": docs})

# b. Generate Answer
def generate_answer(state: IterativeReflectRAGState) -> IterativeReflectRAGState:
    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)
    prompt = f"""
Use the following context to answer the question:

Context:
{context}

Question:
{state.question}
"""
    response = llm.invoke(prompt.strip()).content.strip()
    return state.model_copy(update={"answer": response, "attempts": state.attempts + 1})

# c. Reflect
def reflect_on_answer(state: IterativeReflectRAGState) -> IterativeReflectRAGState:
    prompt = f"""
Evaluate whether the answer below is factually sufficient and complete.

Question: {state.question}
Answer: {state.answer}

Respond 'YES' if it's complete, otherwise 'NO' with feedback.
"""
    feedback = llm.invoke(prompt).content.lower()
    verified = "yes" in feedback
    return state.model_copy(update={"verified": verified})

# d. Refine Query
def refine_query(state: IterativeReflectRAGState) -> IterativeReflectRAGState:
    prompt = f"""
The answer appears incomplete. Suggest a better version of the query that would help retrieve more relevant context.

Original Question: {state.question}
Current Answer: {state.answer}
"""
    new_query = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"refined_question": new_query})

In [8]:
# ---------------------------------
# 5. Build LangGraph
# ---------------------------------
builder = StateGraph(IterativeReflectRAGState)

builder.add_node("retrieve", retrieve_docs)
builder.add_node("answer", generate_answer)
builder.add_node("reflect", reflect_on_answer)
builder.add_node("refine", refine_query)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "answer")
builder.add_edge("answer", "reflect")

builder.add_conditional_edges(
    "reflect",
    lambda s: END if s.verified or s.attempts >= 2 else "refine"
)

builder.add_edge("refine", "retrieve")
builder.add_edge("answer", END)

graph = builder.compile()

In [9]:
# ---------------------------------
# 6. Gradio Interface
# ---------------------------------
def iterative_rag_pipeline(user_query: str):
    init_state = IterativeReflectRAGState(question=user_query)
    result = graph.invoke(init_state)

    return (
        f"Final Answer:\n{result['answer']}\n\n"
        f"Verified: {result['verified']}\n"
        f"Attempts: {result['attempts']}"
    )

demo = gr.Interface(
    fn=iterative_rag_pipeline,
    inputs=gr.Textbox(label="Ask a complex question about the Xumo Manual"),
    outputs=gr.Textbox(label="Iterative + Reflection RAG Output"),
    title="Xumo Manual Iterative + Reflection RAG Assistant"
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
